In [9]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# FASE: Paso 6. Visualización — TOP 3 MODELOS
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

INPUT_DIR  = 'ml_results_grouped'
DATA_DIR   = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_grouped/plots_top3'

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSROOM     = '1.5'
MAX_OCCUPANCY = 30
medals        = ['🥇', '🥈', '🥉']
medal_labels  = ['1st', '2nd', '3rd']
TOP3_COLORS   = ['#2ecc71', '#3498db', '#e67e22']

print("\n" + "="*70)
print(f"AULA {CLASSROOM} — VISUALIZACIÓN TOP 3 MODELOS")
print("="*70)

# ==========================================
# 1. CARGAR RESULTADOS
# ==========================================
print("\n1. LOADING RESULTS...")

df_summary = pd.read_excel(os.path.join(INPUT_DIR, 'models_summary.xlsx'), index_col=0)

# Normalizar nombres de columnas (compatible con ambas versiones del step 5)
col_map = {
    'R2_Test': 'R2', 'RMSE_Test': 'RMSE', 'MAE_Test': 'MAE',
    'R2_Val': 'Val_R2', 'RMSE_Val': 'Val_RMSE',
}
df_summary = df_summary.rename(columns={k: v for k, v in col_map.items() if k in df_summary.columns})
df_summary = df_summary.sort_values('Val_R2' if 'Val_R2' in df_summary.columns else 'R2', ascending=False)

top3 = df_summary.index[:3].tolist()

with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)
with open(os.path.join(INPUT_DIR, 'results.pkl'), 'rb') as f:
    results_raw = pickle.load(f)

df_all_preds = pd.read_excel(os.path.join(INPUT_DIR, 'all_predictions.xlsx'))
y_test = df_all_preds['Actual'].values

print(f"   Top 3: {top3}")

# Calcular métricas completas para los 3
def get_metrics(name):
    y_pred    = np.array(predictions[name])
    errors    = y_test - y_pred
    abs_err   = np.abs(errors)
    mask_nz   = y_test != 0
    return {
        'y_pred':     y_pred,
        'errors':     errors,
        'abs_errors': abs_err,
        'r2':         r2_score(y_test, y_pred),
        'rmse':       np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae':        mean_absolute_error(y_test, y_pred),
        'mape':       np.mean(np.abs((y_test[mask_nz] - y_pred[mask_nz]) / y_test[mask_nz])) * 100,
        'max_err':    np.max(abs_err),
        'std_err':    np.std(errors),
        'acc_exacta': np.mean(abs_err == 0),
        'acc_1':      np.mean(abs_err <= 1),
        'acc_2':      np.mean(abs_err <= 2),
        'acc_3':      np.mean(abs_err <= 3),
    }

metrics = {name: get_metrics(name) for name in top3}

# Imprimir resumen
print("\n   📊 MÉTRICAS TOP 3:")
print(f"   {'R²':>7} {'RMSE':>7} {'MAE':>7} {'Acc±2':>7} {'MAPE':>7}")
print("   " + "─"*62)
for medal, name in zip(medals, top3):
    m = metrics[name]
    print(f"   {medal} {name:<28} {m['r2']:>7.4f} {m['rmse']:>7.2f} {m['mae']:>7.2f} {m['acc_2']:>7.1%} {m['mape']:>7.1f}%")

print("\n2. GENERATING PLOTS...")

# ===================================================================
# ══════════════════════════════════════════════════════════════════
# BLOQUE A — COMPARATIVAS CONJUNTAS
# ══════════════════════════════════════════════════════════════════
# ===================================================================

# ── A1: Predicciones vs Reales (3 subplots) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Predicciones vs Reales — Top 3 Modelos — Aula {CLASSROOM}',
             fontsize=14, fontweight='bold')

for ax, name, color, label in zip(axes, top3, TOP3_COLORS, medal_labels):
    m = metrics[name]
    ax.scatter(y_test, m['y_pred'], alpha=0.7, s=60, color=color,
               edgecolors='white', linewidth=0.5)
    lims = [min(y_test.min(), m['y_pred'].min()) - 1,
            max(y_test.max(), m['y_pred'].max()) + 1]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Ideal')
    ax.set_xlabel('Real (personas)', fontsize=10)
    ax.set_ylabel('Predicho (personas)', fontsize=10)
    ax.set_title(f'{label}: {name}\nRMSE={m["rmse"]:.2f} | R²={m["r2"]:.4f}',
                 fontsize=10, fontweight='bold')
    ax.text(0.05, 0.95, f'MAE={m["mae"]:.2f}\nAcc±2={m["acc_2"]:.1%}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A1_predictions_vs_actual_top3.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ A1_predictions_vs_actual_top3.png")

# ── A2: Comparativa de métricas (barras) ──
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(f'Comparativa de Métricas — Top 3 — Aula {CLASSROOM}',
             fontsize=13, fontweight='bold')

metric_items = [
    ('r2',    'R²',             False),
    ('rmse',  'RMSE (personas)', True),
    ('mae',   'MAE (personas)',  True),
    ('acc_2', 'Acc ±2',         False),
]

short_names = [n.replace(' (', '\n(') for n in top3]

for ax, (key, label, lower_better) in zip(axes, metric_items):
    vals   = [metrics[n][key] for n in top3]
    colors = TOP3_COLORS
    bars   = ax.bar(range(3), vals, color=colors, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(3))
    ax.set_xticklabels(short_names, fontsize=8)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.3f}' if key != 'acc_2' else f'{v:.1%}',
                ha='center', fontsize=9, fontweight='bold')
    if lower_better:
        ax.set_ylabel('↓ mejor', fontsize=9, color='gray')
    else:
        ax.set_ylabel('↑ mejor', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A2_metrics_comparison_top3.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ A2_metrics_comparison_top3.png")

# ── A3: Distribución de errores (violin superpuesto) ──
fig, ax = plt.subplots(figsize=(10, 6))
data    = [metrics[n]['errors'] for n in top3]
labels  = [f'{l}: {n}' for l, n in zip(medal_labels, top3)]
parts   = ax.violinplot(data, positions=range(3), showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], TOP3_COLORS):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.axhline(y=0, color='red', linestyle='--', lw=2, alpha=0.8, label='Error = 0')
ax.set_xticks(range(3))
ax.set_xticklabels(labels, fontsize=9, rotation=10, ha='right')
ax.set_ylabel('Error (personas)', fontsize=11, fontweight='bold')
ax.set_title(f'Distribución de Errores — Top 3 — Aula {CLASSROOM}',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A3_error_distribution_top3.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ A3_error_distribution_top3.png")

# ── A4: Accuracy headcount (barras agrupadas) ──
fig, ax = plt.subplots(figsize=(10, 6))
tols    = ['acc_exacta', 'acc_1', 'acc_2', 'acc_3']
xlabels = ['Exacta', '±1 persona', '±2 personas', '±3 personas']
x       = np.arange(len(tols))
w       = 0.25

for i, (name, color, label) in enumerate(zip(top3, TOP3_COLORS, medal_labels)):
    vals = [metrics[name][t] for t in tols]
    ax.bar(x + (i - 1) * w, vals, w, label=f'{label}: {name}',
           color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=10)
ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title(f'Accuracy Headcount — Top 3 — Aula {CLASSROOM}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A4_accuracy_headcount_top3.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ A4_accuracy_headcount_top3.png")

# ── A5: Error absoluto vs Ocupación real (3 subplots) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Error Absoluto vs Ocupación Real — Top 3 — Aula {CLASSROOM}',
             fontsize=13, fontweight='bold')

for ax, name, color, label in zip(axes, top3, TOP3_COLORS, medal_labels):
    m = metrics[name]
    sc = ax.scatter(y_test, m['abs_errors'], c=m['abs_errors'], cmap='RdYlGn_r',
                    alpha=0.75, s=60, edgecolors='white', linewidth=0.5)
    plt.colorbar(sc, ax=ax, label='Error abs.')
    ax.axhline(y=m['mae'], color='navy', linestyle='--', lw=2,
               label=f'MAE={m["mae"]:.2f}')
    ax.set_xlabel('Real (personas)', fontsize=10)
    ax.set_ylabel('Error Absoluto', fontsize=10)
    ax.set_title(f'{label}: {name}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A5_abs_error_vs_real_top3.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✓ A5_abs_error_vs_real_top3.png")

# ===================================================================
# ══════════════════════════════════════════════════════════════════
# BLOQUE B — DETALLE INDIVIDUAL DE CADA MODELO
# ══════════════════════════════════════════════════════════════════
# ===================================================================

unique_occ = sorted(np.unique(y_test))

for rank, (name, color, label) in enumerate(zip(top3, TOP3_COLORS, medal_labels), 1):
    m     = metrics[name]
    pfx   = f'B{rank}_{label}'
    title = f'{label}: {name} — Aula {CLASSROOM}'

    # ── B_rank_1: Predicciones vs Reales (detalle) ──
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(y_test, m['y_pred'], alpha=0.7, s=80, color=color,
               edgecolors='white', linewidth=0.8)
    lims = [min(y_test.min(), m['y_pred'].min()) - 1,
            max(y_test.max(), m['y_pred'].max()) + 1]
    ax.plot(lims, lims, 'k--', lw=2, alpha=0.7, label='Ideal')
    ax.set_xlabel('Ocupación Real (personas)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Ocupación Predicha (personas)', fontsize=12, fontweight='bold')
    ax.set_title(f'{title}\nPredicciones vs Reales', fontsize=12, fontweight='bold')
    ax.text(0.05, 0.95,
            f'R² = {m["r2"]:.4f}\nRMSE = {m["rmse"]:.2f}\nMAE = {m["mae"]:.2f}\nAcc±2 = {m["acc_2"]:.1%}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{pfx}_1_predictions.png'), dpi=200, bbox_inches='tight')
    plt.close()

    # ── B_rank_2: Histograma de errores ──
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(m['errors'], bins=15, alpha=0.7, color=color, edgecolor='white', density=True)
    mu, sigma = stats.norm.fit(m['errors'])
    x_r = np.linspace(m['errors'].min(), m['errors'].max(), 200)
    ax.plot(x_r, stats.norm.pdf(x_r, mu, sigma), 'r-', lw=2,
            label=f'Normal (μ={mu:.2f}, σ={sigma:.2f})')
    ax.axvline(x=0, color='black', linestyle='--', lw=2, alpha=0.7, label='Error=0')
    ax.set_xlabel('Error (personas)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Densidad', fontsize=12, fontweight='bold')
    ax.set_title(f'{title}\nDistribución de Errores', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{pfx}_2_error_histogram.png'), dpi=200, bbox_inches='tight')
    plt.close()

    # ── B_rank_3: Error por nivel de ocupación (boxplot) ──
    fig, ax = plt.subplots(figsize=(10, 6))
    data_by_occ = [m['abs_errors'][y_test == occ] for occ in unique_occ]
    bp = ax.boxplot(data_by_occ, positions=unique_occ, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(y=m['mae'], color='red', linestyle='--', lw=2,
               label=f'MAE global = {m["mae"]:.2f}')
    ax.set_xlabel('Ocupación Real (personas)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Error Absoluto (personas)', fontsize=11, fontweight='bold')
    ax.set_title(f'{title}\nError por Nivel de Ocupación', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{pfx}_3_error_by_occ.png'), dpi=200, bbox_inches='tight')
    plt.close()

    # ── B_rank_4: Q-Q plot ──
    fig, ax = plt.subplots(figsize=(7, 7))
    stats.probplot(m['errors'], dist="norm", plot=ax)
    ax.set_title(f'{title}\nQ-Q Plot de Errores', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{pfx}_4_qqplot.png'), dpi=200, bbox_inches='tight')
    plt.close()

    print(f"   ✓ {pfx}_1/2/3/4 ({name})")

# ===================================================================
# GUARDAR TABLA RESUMEN
# ===================================================================
print("\n3. SAVING SUMMARY TABLE...")

rows = []
for medal, label, name in zip(medals, medal_labels, top3):
    m = metrics[name]
    rows.append({
        'Rank': f'{medal} {label}',
        'Modelo': name,
        'R²': round(m['r2'], 4),
        'RMSE': round(m['rmse'], 2),
        'MAE': round(m['mae'], 2),
        'MAPE_%': round(m['mape'], 2),
        'Max_Error': round(m['max_err'], 1),
        'Std_Error': round(m['std_err'], 2),
        'Acc_Exacta': f"{m['acc_exacta']:.1%}",
        'Acc_±1':     f"{m['acc_1']:.1%}",
        'Acc_±2':     f"{m['acc_2']:.1%}",
        'Acc_±3':     f"{m['acc_3']:.1%}",
    })

df_top3 = pd.DataFrame(rows)
df_top3.to_excel(os.path.join(OUTPUT_DIR, 'top3_metrics_summary.xlsx'), index=False)
print("   ✓ top3_metrics_summary.xlsx")

# ===================================================================
# RESUMEN FINAL
# ===================================================================
print("\n" + "="*70)
print("✅ VISUALIZACIONES TOP 3 COMPLETADAS")
print("="*70)
print(f"\n   📁 {OUTPUT_DIR}/")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")

print(f"\n   🥇 {top3[0]}: R²={metrics[top3[0]]['r2']:.4f} | RMSE={metrics[top3[0]]['rmse']:.2f}")
print(f"   🥈 {top3[1]}: R²={metrics[top3[1]]['r2']:.4f} | RMSE={metrics[top3[1]]['rmse']:.2f}")
print(f"   🥉 {top3[2]}: R²={metrics[top3[2]]['r2']:.4f} | RMSE={metrics[top3[2]]['rmse']:.2f}")


AULA 1.5 — VISUALIZACIÓN TOP 3 MODELOS

1. LOADING RESULTS...
   Top 3: ['SVR (linear, C=0.5)', 'Ridge (α=10)', 'Lasso (α=1)']

   📊 MÉTRICAS TOP 3:
        R²    RMSE     MAE   Acc±2    MAPE
   ──────────────────────────────────────────────────────────────
   🥇 SVR (linear, C=0.5)           0.2196   14.41   10.37   17.8%    70.5%
   🥈 Ridge (α=10)                  0.0731   15.71   12.04   13.3%    81.3%
   🥉 Lasso (α=1)                   0.1418   15.12   11.15   17.8%    75.4%

2. GENERATING PLOTS...
   ✓ A1_predictions_vs_actual_top3.png
   ✓ A2_metrics_comparison_top3.png
   ✓ A3_error_distribution_top3.png
   ✓ A4_accuracy_headcount_top3.png
   ✓ A5_abs_error_vs_real_top3.png
   ✓ B1_1st_1/2/3/4 (SVR (linear, C=0.5))
   ✓ B2_2nd_1/2/3/4 (Ridge (α=10))
   ✓ B3_3rd_1/2/3/4 (Lasso (α=1))

3. SAVING SUMMARY TABLE...
   ✓ top3_metrics_summary.xlsx

✅ VISUALIZACIONES TOP 3 COMPLETADAS

   📁 ml_results_grouped/plots_top3/
      A1_predictions_vs_actual_top3.png (185 KB)
      A2_metrics_